# 1. Imports

In [1]:
import pandas as pd
from pathlib import Path
import os

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import *

import plotly.graph_objects as go

pd.set_option('display.max_columns', None)

# 2. Funções

Funções auxiliares reaproveitadas mais adiante: leitura dos parquets de uma temporada, cálculo de métricas de ameaça por zona do campo, e plot de validação de um evento.

## 2.1. Função para pegar os eventos de uma temporada nos arquivos parquet

In [2]:
def get_season_events_parquet_file_paths(events_competition_season_folder_path):
    
    season_events_parquet_file_paths = [
        str(Path(events_competition_season_folder_path) / season_event_parquet_file) 
        for season_event_parquet_file in os.listdir(events_competition_season_folder_path) 
        if season_event_parquet_file.endswith('.parquet')
        ]
    
    return season_events_parquet_file_paths

## 2.2. Funções para métricas de ameaça por zona do campo

Define, para cada zona do campo (full/half/third_2/third_3), a contagem de atacantes/defensores entre a bola e o gol e a distância de progressão — usadas depois na criação do `threat_score`.

In [3]:
# ============================================================
# Funções reutilizáveis para cálculo por zona do campo (eixo x)
# ============================================================

# Cada zona é definida só pelo x_start (ponto de corte) — sempre [x_start, right_x].
# 'full' usa x_start = left_x (equivale a nunca mascarar, já que ball_x >= left_x sempre).

def get_zone_x_start(zone, left_x, right_x):
    """
    Retorna o x_start (ponto de corte) de onde a zona passa a valer.
    A zona é sempre [x_start, right_x] — jogadores/distância considerados
    dali até o gol.
    """
    field_length = right_x - left_x

    if zone == 'full':
        return left_x
    elif zone == 'half':
        return F.lit(0.0)
    elif zone == 'third_2':
        return left_x + field_length / 3
    elif zone == 'third_3':
        return left_x + 2 * field_length / 3
    else:
        raise ValueError(f"Zona inválida: {zone}")


def zone_col_names(zone):
    """Padroniza os nomes de coluna de cada zona (zona 'full' não leva sufixo)."""
    suffix = '' if zone == 'full' else f'_{zone}'
    return {
        'attackers':   f'attackers{suffix}',
        'defenders':   f'defenders{suffix}',
        'total':       f'total_players{suffix}',
        'advantage':   f'atk_def_advantage{suffix}',
        'progression': f'progression_dist{suffix}',
    }


def add_zone_metrics(df, ball_x, ball_y, left_x, right_x, top_y, bottom_y, zones):
    """
    Para cada zona [x_start, right_x]:
      - se a bola estiver dentro (ball_x >= x_start): calcula total_players,
        atk_def_advantage e progression_dist, senão: NULL

    attackers/defenders/total/advantage: contam jogadores entre a bola
    (ball_x) e o gol (right_x), em todas as zonas — a zona só decide
    se esse valor é reportado (mask) ou fica NULL para aquele evento.
    Quando a bola já passou da linha de entrada da área mas está fora
    dela lateralmente (fora do intervalo de y da área), a contagem passa
    a considerar só os jogadores dentro do retângulo da área (x e y).
    Se a bola estiver dentro da área, ou ainda não tiver alcançado a
    linha, mantém o corredor original (só eixo x, entre bola e gol).

    progression_dist: usa x_start como o "escanteio" simulado da zona
    (em vez do escanteio real do campo, exceto na zona 'full', onde
    x_start == left_x, ou seja, o escanteio real).
    """
    # Retângulo da área: 16.5m de profundidade a partir do gol, 40.3m de largura
    # (largura fixa por regulamento, centrada no gol — não depende de stadiumWidth)
    area_left_x = right_x - 16.5
    area_right_x = right_x
    area_top_y = F.lit(20.15)
    area_bottom_y = F.lit(-20.15)

    ball_past_area_line = ball_x >= area_left_x
    ball_outside_area_y = (ball_y > area_top_y) | (ball_y < area_bottom_y)

    # só troca pra contagem por retângulo se a bola já passou da linha
    # E está fora da área lateralmente — dentro da área, mantém o corredor
    use_area_count = ball_past_area_line & ball_outside_area_y

    def player_in_scope(p):
        return F.when(
            use_area_count,
            (p['x'] >= area_left_x) & (p['x'] <= area_right_x) &
            (p['y'] <= area_top_y) & (p['y'] >= area_bottom_y)
        ).otherwise(
            (p['x'] <= right_x) & (p['x'] >= ball_x)
        )
    
    def euclidean_dist(x1, y1, x2, y2):
        return F.sqrt(
            F.pow(x1 - x2, 2) +
            F.pow(y1 - y2, 2)
        )

    for zone in zones:
        x_start = get_zone_x_start(zone, left_x, right_x)
        cols = zone_col_names(zone)

        ball_in_zone = ball_x >= x_start

        attackers = F.size(F.filter(F.col('attackingPlayersNorm'), player_in_scope))
        defenders = F.size(F.filter(F.col('defendingPlayersNorm'), player_in_scope))
        total = attackers + defenders
        advantage = attackers - defenders

        progression = F.round(
            F.least(
                euclidean_dist(ball_x, ball_y, x_start, top_y),
                euclidean_dist(ball_x, ball_y, x_start, bottom_y)
            ), 2
        )

        df = df.withColumns({
            cols['attackers']:   F.when(ball_in_zone, attackers).otherwise(F.lit(None)),
            cols['defenders']:   F.when(ball_in_zone, defenders).otherwise(F.lit(None)),
            cols['total']:       F.when(ball_in_zone, total).otherwise(F.lit(None)),
            cols['advantage']:   F.when(ball_in_zone, advantage).otherwise(F.lit(None)),
            cols['progression']: F.when(ball_in_zone, progression).otherwise(F.lit(None)),
        })

    return df

## 2.3. Função de plot para validação visual de um evento

Desenha jogadores, bola e as métricas de ameaça de um único evento — usada só pra inspeção manual/sanity check, não faz parte do pipeline de dados.

In [4]:
def plot_threat_event(df_threat, show_player_names=False, show_player_positions=False):
    """
    Plota um evento para validar:
    - attackers_between_ball_goal[_zona]
    - defenders_between_ball_goal[_zona]
    - total_players[_zona]
    - atk_def_advantage[_zona]

    Espera um DataFrame Spark contendo exatamente um evento.

    Sempre desenha: linha da bola, linha do meio-campo (x=0) e as
    2 linhas que dividem o campo em 3 terços.
    """

    pdf = df_threat.toPandas()

    if len(pdf) != 1:
        raise ValueError("O dataframe deve conter exatamente um evento.")

    row = pdf.iloc[0]

    attackers = row["attackingPlayersNorm"]
    defenders = row["defendingPlayersNorm"]
    ball = row["ballsNorm"][0]

    stadium_length = row["stadiumLength"]
    stadium_width = row["stadiumWidth"]

    left_x = -stadium_length / 2
    right_x = stadium_length / 2
    bottom_y = -stadium_width / 2
    top_y = stadium_width / 2

    ball_x = ball["x"]

    show_labels = show_player_names or show_player_positions

    def player_label(p):
        parts = []
        if show_player_names:
            parts.append(p["player"]["name"])
        if show_player_positions:
            parts.append(f'({p["position"]})')
        return " ".join(parts) if parts else None

    fig = go.Figure()

    # ============================
    # Atacantes
    # ============================

    fig.add_trace(
    go.Scatter(
        x=[p["x"] for p in attackers],
        y=[p["y"] for p in attackers],
        mode="markers+text" if show_labels else "markers",
        text=[player_label(p) for p in attackers] if show_labels else None,
        textposition="top center",
        marker=dict(
            size=10,
            color=[
                "red" if p["x"] >= ball_x else "lightcoral"
                for p in attackers
            ]
        ),
        name="Attackers"
    )
)

    # ============================
    # Defensores
    # ============================

    fig.add_trace(
    go.Scatter(
        x=[p["x"] for p in defenders],
        y=[p["y"] for p in defenders],
        mode="markers+text" if show_labels else "markers",
        text=[player_label(p) for p in defenders] if show_labels else None,
        textposition="top center",
        marker=dict(
            size=10,
            color=[
                "blue" if p["x"] >= ball_x else "lightblue"
                for p in defenders
            ]
        ),
        name="Defenders"
    )
)

    # ============================
    # Bola
    # ============================

    fig.add_trace(
        go.Scatter(
            x=[ball["x"]],
            y=[ball["y"]],
            mode="markers",
            marker=dict(
                color="black",
                size=10,
                symbol="circle"
            ),
            name="Ball"
        )
    )

    # ============================
    # Linha da bola
    # ============================

    fig.add_vline(
        x=ball_x,
        line_dash="dash",
        line_width=2,
        line_color="black"
    )

    # ============================
    # Linhas divisórias fixas: meio-campo + 2 terços
    # ============================

    field_length = right_x - left_x

    zone_boundaries = [
        0.0,                                # meio-campo (half)
        #left_x + field_length / 3,          # início do terço 2
        #left_x + 2 * field_length / 3,      # início do terço 3
    ]

    for x_boundary in zone_boundaries:
        fig.add_vline(
            x=x_boundary,
            line_dash="dot",
            line_width=2,
            line_color="gray"
        )

    # ============================
    # Limites do campo
    # ============================

    fig.update_xaxes(
        range=[left_x, right_x],
        title="X",
        zeroline=False
    )

    fig.update_yaxes(
        range=[bottom_y, top_y],
        title="Y",
        scaleanchor="x",
        scaleratio=1,
        zeroline=False
    )

    # ============================
    # Layout
    # ============================
    height = 600
    width = int(height * stadium_length / stadium_width)

    fig.update_layout(
    template="simple_white",
    width=width,
    height=height,
    margin=dict(l=20, r=120, t=60, b=20),
    title=(
        f"Evento: {row['eventTypeDescription']} | "
        f"Posse: {row['eventTeamName']} | "
        f"P. Invertida: {row['flipped_homeTeam']} | "
        f"Ameaça: {row['threat_score']:.3f} | "
        f"Impacto: {row['threat_score_impact']:.3f}"
    ),
    legend=dict(
        x=1.02,
        y=1,
        xanchor="left",
        yanchor="top",
        orientation="v",
        bgcolor="rgba(255,255,255,0.8)"
    )
    )

    fig.show()

# 3. Preparação dos dados

Carrega os eventos de posse (parquet) e os jogos (csv) de uma competição/temporada, limpa e junta as duas fontes numa única base.

## 3.1. Criação da Sessão Spark

In [5]:
# Criação da sessão Spark local
spark = (
    SparkSession
    .builder
    .config("spark.driver.memory", "4g") 
    .config("spark.executor.memory", "4g") 
    .master("local[*]")
    .appName("target_engineering")
    .getOrCreate()
    )

## 3.2. Criação de df com eventos de todas as partidas da temporada de 2022-2023 da Premier League

In [6]:
# competition_id = 1 (Premier League)
# season = 2022-2023
events_competition_season_folder_path = str(Path().resolve().parent.parent / "data" / "events" / "1" / "2022-2023")

season_events_parquet_file_paths = get_season_events_parquet_file_paths(events_competition_season_folder_path)

# Criação do dataframe concatenando todos os arquivos parquet dos eventos das partidas entre as temporadas de todas as competições
df_events = spark.read.parquet(*season_events_parquet_file_paths)

df_events.show()

+--------------------+-------------+------+---------+------+-----------------+------------+--------------------+--------------+-----------------------+--------+--------------------+--------------------+--------------------+--------------------+-------------+----------------+-----------+--------------+-----------------------+-----------------------+
|             eventId|competitionId|gameId|   season|period|periodDescription|   eventType|eventTypeDescription|startGameClock|startFormattedGameClock|homeTeam|      details_parsed|  homePlayers_parsed|  awayPlayers_parsed|        balls_parsed|eventPlayerId| eventPlayerName|eventTeamId| eventTeamName|eventSubTypeDescription|eventOutcomeDescription|
+--------------------+-------------+------+---------+------+-----------------+------------+--------------------+--------------+-----------------------+--------+--------------------+--------------------+--------------------+--------------------+-------------+----------------+-----------+-----------

In [7]:
print('Quantidade de eventos na temporada:', df_events.select('eventId').count())
print('Quantidade de eventos duplicados na temporada:', df_events.select('eventId').distinct().count())

Quantidade de eventos na temporada: 945154
Quantidade de eventos duplicados na temporada: 944838


In [8]:
df_events = df_events.drop_duplicates(subset=['eventId'])

In [9]:
df_events = (
    df_events
    # remoção de tipos de eventos reduntantes, desnecessários ou incosistentes
    .filter(
        ~F.col('eventTypeDescription').isin([
            'A possession with a player on the ball',
            'Unknown',
            'Substitution',
            'Ball hits the woodwork or corner flag and comes back into play', 
            'Player comes off the pitch'
            ])
    )
)

## 3.3. Obter jogos da temporada e ajuste da identificação do mandante/adversário

In [10]:
games_path = str(Path().resolve().parent.parent / "data" / "games.csv")

df_games_raw = spark.read.csv(games_path, header=True)
df_games_raw = df_games_raw.filter(F.col('season') == '2022-2023')
df_games_raw.show()

+------+----------+---------+----------------------+-------------+-------------+------+--------------------+-------------+---------------+--------------+--------------------+--------------------+-------------+------------+
|gameId|      date|   season|teamExtraTimeStartSide|teamStartSide|    venueType|teamId|            teamName|competitionId|competitionName|opponentTeamId|    opponentTeamName|         stadiumName|stadiumLength|stadiumWidth|
+------+----------+---------+----------------------+-------------+-------------+------+--------------------+-------------+---------------+--------------+--------------------+--------------------+-------------+------------+
|  4786|2023-05-13|2022-2023|                 Right|         Left|    TEAM_HOME|     3|         Aston Villa|            1| Premier League|            17|   Tottenham Hotspur|          Villa Park|        105.0|        68.0|
|  4614|2022-12-30|2022-2023|                  Left|        Right|OPPONENT_HOME|   119|           Brentford|

In [11]:
# filtro para considerar apenas os jogos que tiverem um mandante/adversário definido
# pois foi visto no sanity 'tracking_events_one_season' que existem casos de venueType neutro e não serão considerados para não atrapalhar nos cálculos futuros 
df_games_raw = df_games_raw.filter(F.col('venueType').isin(['TEAM_HOME', 'OPPONENT_HOME']))

In [12]:
# se venueType == TEAM_HOME, (homeTeamId == teamId e homeTeamName == teamName) e (opponentTeamId == opponentTeamId e opponentTeamName == opponentTeamName)
# se venueType == OPPONENT_HOME, (homeTeamId == opponentTeamId e homeTeamName == opponentTeamName) e (opponentTeamId == teamId e opponentTeamName == teamName)
df_games = (
    df_games_raw
    .withColumns({
        # homeTeam = "team" quando o mandante é o "team" (TEAM_HOME), senão homeTeam = "opponentTeam"
        "homeTeamId": F.when(F.col("venueType") == "TEAM_HOME", F.col("teamId")).otherwise(F.col("opponentTeamId")),
        "homeTeamName": F.when(F.col("venueType") == "TEAM_HOME", F.col("teamName")).otherwise(F.col("opponentTeamName")),
        # homeTeamStartSide = lado que o time mandante começou:
        # - venueType == TEAM_HOME: mandante é o "team" -> usa teamStartSide direto
        # - venueType == OPPONENT_HOME: mandante é o "opponentTeam" (lado não vem direto na base) ->
        #   usa o complementar do teamStartSide (Right vira Left e vice-versa)
        "homeTeamStartSide": F.when(
            F.col("venueType") == "TEAM_HOME", F.col("teamStartSide")
            ).otherwise(
                F.when(F.col("teamStartSide") == "Right", F.lit("Left")).otherwise(F.lit("Right"))),
        
        # opponentTeam = "opponentTeam" quando o mandante é o "team" (TEAM_HOME), senão opponentTeam = "team"
        "opponentTeamId": F.when(F.col("venueType") == "TEAM_HOME", F.col("opponentTeamId")).otherwise(F.col("teamId")),
        "opponentTeamName": F.when(F.col("venueType") == "TEAM_HOME", F.col("opponentTeamName")).otherwise(F.col("teamName")),
        # opponentTeamStartSide = lado que o time visitante começou:
        # - venueType == TEAM_HOME: visitante é o "opponentTeam" (lado não vem direto na base) ->
        #   usa o complementar do teamStartSide
        # - venueType == OPPONENT_HOME: visitante é o próprio "team" -> usa teamStartSide direto
        "opponentTeamStartSide": F.when(
            F.col("venueType") == "TEAM_HOME", 
            F.when(F.col("teamStartSide") == "Right", F.lit("Left")).otherwise(F.lit("Right"))
            ).otherwise(F.col("teamStartSide")),
    })
    .select(
        'gameId',
        'competitionId',
        'competitionName',
        'date',
        'season',
        'venueType',
        #'homeTeamId',
        'homeTeamName',
        #'opponentTeamId',
        'opponentTeamName',
        'homeTeamStartSide',
        'opponentTeamStartSide',
        'stadiumName',
        F.col('stadiumLength').cast("float"),
        F.col('stadiumWidth').cast("float")
    )
)

df_games.show(5)

+------+-------------+---------------+----------+---------+-------------+--------------------+--------------------+-----------------+---------------------+--------------------+-------------+------------+
|gameId|competitionId|competitionName|      date|   season|    venueType|        homeTeamName|    opponentTeamName|homeTeamStartSide|opponentTeamStartSide|         stadiumName|stadiumLength|stadiumWidth|
+------+-------------+---------------+----------+---------+-------------+--------------------+--------------------+-----------------+---------------------+--------------------+-------------+------------+
|  4786|            1| Premier League|2023-05-13|2022-2023|    TEAM_HOME|         Aston Villa|   Tottenham Hotspur|             Left|                Right|          Villa Park|        105.0|        68.0|
|  4614|            1| Premier League|2022-12-30|2022-2023|OPPONENT_HOME|            West Ham|           Brentford|             Left|                Right|      London Stadium|        

# 4. Junção dos dados dos Jogos + Eventos em uma tabela

Junta eventos e jogos, valida a direção de ataque de cada time, cria o `possession_id` (com a correção especial de Clearance) e enriquece o tracking dos jogadores com a posição de cada um.

## 4.1. Join eventos + jogos e ajuste de lado por período

Junta eventos e jogos por `(competitionId, season, gameId)`, remove jogos de mandante neutro, e deriva o sentido de ataque de cada time (sempre o lado onde o adversário começou o período).

In [13]:
# left join dos eventos + informações dos jogos
df_games_events = (
    df_events
    .join(
        df_games, 
        on = ["competitionId", "season", "gameId"],
        how='left'
    )
    # filtro para remover os jogos que tinha mandante neutro (venueType == NEUTRAL)
    .filter(~F.col('date').isNull())
)

df_games_events = (
    df_games_events
    # considerar a reversão de lado conforme mudança do primero para o segundo tempo
    # se for primeiro tempo, mantém a variável de StartSide, se não é o contrário
    .withColumns({
        'homeTeamStartSide': F.when(F.col('period') == 1, F.col('homeTeamStartSide')).otherwise(F.col('opponentTeamStartSide')),
        'opponentTeamStartSide': F.when(F.col('period') == 1, F.col('opponentTeamStartSide')).otherwise(F.col('homeTeamStartSide'))
    }) 

    # Sentido do ataque do time é sempre o lado que o outro time começou o período
    .withColumns({
        'homeTeamAttackDirection': F.col('opponentTeamStartSide'),
        'awayTeamAttackDirection': F.col('homeTeamStartSide')
    })
    #.drop('homeTeamStartSide')
)

df_games_events.show(5)

+-------------+---------+------+--------------------+------+-----------------+---------+--------------------+--------------+-----------------------+--------+--------------------+--------------------+--------------------+--------------------+-------------+-----------------+-----------+--------------------+-----------------------+-----------------------+---------------+----------+-------------+------------+--------------------+-----------------+---------------------+----------------+-------------+------------+-----------------------+-----------------------+
|competitionId|   season|gameId|             eventId|period|periodDescription|eventType|eventTypeDescription|startGameClock|startFormattedGameClock|homeTeam|      details_parsed|  homePlayers_parsed|  awayPlayers_parsed|        balls_parsed|eventPlayerId|  eventPlayerName|eventTeamId|       eventTeamName|eventSubTypeDescription|eventOutcomeDescription|competitionName|      date|    venueType|homeTeamName|    opponentTeamName|homeTeamS

## 4.2. Criação do Id de posse e correção do mandante em Clearance

Gera `possession_id` (muda toda vez que `homeTeam` alterna) e `last_cycle_event` (último evento de cada posse). Antes disso, corrige o `homeTeam` bruto de eventos de Clearance — que não indicam de forma confiável quem ficou com a posse — olhando pro time do próximo evento não-Clearance.

In [14]:
# window function pra criação do Id de posse
w_pos = (
    Window
    .partitionBy(
        "competitionId",
        "season",
        "gameId"
    )
    .orderBy("startGameClock")
)

df_games_events = (
    df_games_events
    .filter(
        # filtro para não considerar tracking da bola e jogadores home/away que n tem tracking (dps podemos pensar em imputar)
        (F.size("balls_parsed") != 0) & (F.size("homePlayers_parsed") != 0) & (F.size("awayPlayers_parsed") != 0)
    ) 
    # drop nos eventos onde nenhum dos dois times tem a posse
    .dropna(subset='homeTeam')
    # Preserva o homeTeam bruto pra montar a flag de flip logo abaixo
    .withColumn('homeTeamOriginal', F.col('homeTeam'))
    
)

w_events = Window.partitionBy('competitionId', 'season', 'gameId').orderBy('startGameClock')

# Corrige o homeTeam da Clearance ANTES de calcular possession_id.
# seed_team = homeTeam do evento, exceto Clearance (NULL, já que seu homeTeam bruto não é confiável — é uma exceção de evento defensivo com posse).
seed_team = F.when(F.col('eventTypeDescription') != 'Clearance', F.col('homeTeam'))

w_fwd = w_events.rowsBetween(Window.currentRow, Window.unboundedFollowing)

# resolved_team olha pra frente e pega o primeiro seed não nulo, ou seja, o
# time do próximo evento não-Clearance: se for do adversário (transição), a
# Clearance flipa pra ele; se for do mesmo time (retenção), fica como já era.
resolved_team = F.first(seed_team, ignorenulls=True).over(w_fwd)

df_games_events = df_games_events.withColumns({
    'homeTeam': F.coalesce(resolved_team, F.col('homeTeam')),
})

# Flag auxiliar: indica se essa Clearance teve o homeTeam corrigido (transição) ou não (retenção)
df_games_events = (
    df_games_events
    .withColumn(
        'flipped_homeTeam', 
        F.col('homeTeam') != F.col('homeTeamOriginal')
    )
    .withColumn(
        "possession_id",
        F.sum(
            F.when(
                F.col("homeTeam") != F.lag("homeTeam").over(w_pos), 1
            ).otherwise(0)
        ).over(w_pos.rowsBetween(Window.unboundedPreceding, Window.currentRow))
    )
    .withColumn(
        "last_cycle_event",
        F.lead("possession_id", 1).over(w_pos).isNull() |
        (F.lead("possession_id", 1).over(w_pos) != F.col("possession_id"))
    )
    .drop(
         'homeTeamOriginal',         
        )
)

#df_games_events.show()

## 4.3. Enriquecimento do tracking com a posição de cada jogador

Junta `games_players.csv` (extract_games_players_info.py) aos eventos por `gameId`.
Como esse dataset é minúsculo perto da tabela de eventos (poucas linhas por jogo),
montamos um mapa `playerId -> playerPosition` por partida e fazemos um broadcast
join, evitando explodir as arrays de tracking (homePlayers_parsed/awayPlayers_parsed)
— o `F.transform` + `element_at` roda inteiramente map-side, sem shuffle na tabela grande.

In [15]:
games_players_path = str(Path().resolve().parent.parent / "data" / "games_players.csv")

df_games_players_raw = (
    spark.read.csv(games_players_path, header=True)
    .withColumnRenamed("player.id", "playerId")
)

# Struct aninhado (type/typeDescription/groupType) por jogador, igual ao
# "player" (id/name) — playerPosition já é o "type"; as outras 2 colunas
# vêm do enriquecimento em extract_games_players_info.py (baseado em positions.md)
df_positions_map = (
    df_games_players_raw
    .withColumns({
        "gameId": F.col("gameId").cast("int"),
        "playerId": F.col("playerId").cast("int"),
    })
    .withColumn(
        "positionInfo",
        F.struct(
            F.col("playerPosition").alias("type"),
            F.col("playerPositionTypeDescription").alias("typeDescription"),
            F.col("playerPositionGroupType").alias("groupType"),
        )
    )
    .groupBy("gameId")
    .agg(
        # Mapa playerId -> positionInfo por jogo (poucas linhas por gameId,
        # cabe tranquilamente num broadcast)
        F.map_from_entries(F.collect_list(F.struct("playerId", "positionInfo"))).alias("positionsMap")
    )
)

# Broadcast join por gameId (tabela de posições é pequena) + transform
# map-side pra adicionar "position" (agora um struct, não mais string) em
# cada elemento de jogador, sem explodir as arrays de tracking. Também
# aproveita o mesmo positionsMap pra trazer eventPlayerPositionType/Group —
# a posição do jogador principal do evento (mesmo padrão de eventPlayerId/
# eventTeamId), antes de descartar a coluna do join.
event_player_position = F.element_at(F.col("positionsMap"), F.col("eventPlayerId").cast("int"))

df_games_events = (
    df_games_events
    .join(F.broadcast(df_positions_map), on="gameId", how="left")
    .withColumns({
        "homePlayers_parsed": F.transform(
            "homePlayers_parsed",
            lambda p: p.withField("position", F.element_at(F.col("positionsMap"), p["player"]["id"]))
        ),
        "awayPlayers_parsed": F.transform(
            "awayPlayers_parsed",
            lambda p: p.withField("position", F.element_at(F.col("positionsMap"), p["player"]["id"]))
        ),
        "eventPlayerPositionType": event_player_position["type"],
        "eventPlayerPositionGroup": event_player_position["groupType"],
    })
    .drop("positionsMap")
)

df_positions_map.show(5, truncate=False)
df_games_events.select('eventPlayerId', 'eventPlayerName', 'eventPlayerPositionType', 'eventPlayerPositionGroup').show(10, truncate=False)

+------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|gameId|positionsMap                                         

In [16]:
df_games_events.show()

+------+-------------+---------+--------------------+------+-----------------+------------+--------------------+--------------+-----------------------+--------+--------------------+--------------------+--------------------+--------------------+-------------+------------------+-----------+--------------+-----------------------+-----------------------+---------------+----------+-------------+--------------+----------------+-----------------+---------------------+-------------+-------------+------------+-----------------------+-----------------------+----------------+-------------+----------------+-----------------------+------------------------+
|gameId|competitionId|   season|             eventId|period|periodDescription|   eventType|eventTypeDescription|startGameClock|startFormattedGameClock|homeTeam|      details_parsed|  homePlayers_parsed|  awayPlayers_parsed|        balls_parsed|eventPlayerId|   eventPlayerName|eventTeamId| eventTeamName|eventSubTypeDescription|eventOutcomeDescript

# 5. Normalização espacial do ataque (sempre para a direita)

A partir daqui, cada evento passa a ter só `attackingPlayers`/`defendingPlayers` (em vez de home/away) e as coordenadas de jogadores e bola são espelhadas no eixo x quando o time com a posse ataca para a esquerda — assim toda métrica geométrica seguinte pode assumir que o ataque é sempre para a direita.

In [17]:
# time com a posse está atacando e time sem está defendendo
df_games_events_tracking = (
    df_games_events
    # time atacando = se o time da casa tiver a posse, pega tracking home, se não pega tracking away
    .withColumn(
        "attackingPlayers",
        F.when(F.col("homeTeam"), F.col("homePlayers_parsed"))
        .otherwise(F.col("awayPlayers_parsed"))
    )
    # time defendendo = se o time da casa tiver a posse, pega tracking away, se não pega tracking home
    .withColumn(
        "defendingPlayers",
        F.when(F.col("homeTeam"), F.col("awayPlayers_parsed"))
        .otherwise(F.col("homePlayers_parsed"))
    )
    .withColumn(
        "attackingDirection",
        F.when(F.col("homeTeam"), F.col("homeTeamAttackDirection"))
        .otherwise(F.col("awayTeamAttackDirection"))
    )
    # flag para normalização (para tratar ataque sempre pra direita)
    .withColumn(
        "need_side_revert",
        F.col("attackingDirection") == "Left"
    )
    .drop(
        'homePlayers_parsed',  
        'awayPlayers_parsed',
        'homeTeamAttackDirection',
        'awayTeamAttackDirection'
    )
)

#df_games_events_tracking.show()

In [18]:
players_tracking_norm = (
        lambda p: F.struct(
            # reversão do eixo horizontal qnd necessário
            F.when(F.col("need_side_revert"), -p["x"])
            .otherwise(p["x"])
            .alias("x"),
            # variáveis restantes mantém igual
            p["y"].alias("y"),            
            p["player"].alias("player"),
            p["visibility"].alias("visibility"),
            p["confidence"].alias("confidence"),
            p["position"].alias("position")
        )
)

balls_norm = (
    F.transform(
        "balls_parsed",
        lambda b: F.struct(
            # reversão do eixo horizontal qnd necessário
            F.when(F.col("need_side_revert"), -b["x"])
            .otherwise(b["x"])
            .alias("x"),
            # variáveis restantes mantém igual            
            b["y"].alias("y"),
            b["z"].alias("z"),            
            b["visibility"].alias("visibility")
        )
    )
)

# df para normalizar os atacantes, defensores e bola sempre atacando do lado direito
df_games_events_tracking_norm = (
    df_games_events_tracking
    # normalizar atacantes
    .withColumns({
        "attackingPlayersNorm": F.transform("attackingPlayers", players_tracking_norm),
        # normalizar defensores
        "defendingPlayersNorm": F.transform("defendingPlayers", players_tracking_norm),
        # normalizar a bola
        "ballsNorm": balls_norm
    }).drop(
        'attackingPlayers', 
        'defendingPlayers',
        'balls_parsed',
        'attackingDirection',
        'need_side_revert'
        )
)

In [19]:
df_games_events_tracking_norm.select('attackingPlayersNorm').first()

Row(attackingPlayersNorm=[Row(x=2.2179999351501465, y=25.923999786376953, player=Row(id=480, name='Wilfried Zaha'), visibility='ESTIMATED', confidence='LOW', position=Row(type='LW', typeDescription='Left Winger', groupType='A')), Row(x=-8.996999740600586, y=-13.696999549865723, player=Row(id=43, name='Nathaniel Clyne'), visibility='ESTIMATED', confidence='LOW', position=Row(type='RWB', typeDescription='Right Wing-Back', groupType='D')), Row(x=0.21199999749660492, y=-10.461999893188477, player=Row(id=483, name='Jordan Ayew'), visibility='ESTIMATED', confidence='LOW', position=Row(type='CF', typeDescription='Centre-Forward', groupType='A')), Row(x=-8.791999816894531, y=20.780000686645508, player=Row(id=581, name='Tyrick Mitchell'), visibility='ESTIMATED', confidence='LOW', position=Row(type='LWB', typeDescription='Left Wing-Back', groupType='D')), Row(x=5.426000118255615, y=21.761999130249023, player=Row(id=6016, name='Odsonne Edouard'), visibility='ESTIMATED', confidence='LOW', position

# 6. Componentes de ameaça de gol (threat score)

Combina 3 métricas geométricas (distância percorrida, jogadores entre bola e gol, vantagem numérica) numa métrica única de ameaça, calculada em 4 zonas do campo (full/half/third_2/third_3).

In [20]:
# Extremos esquerdo e direito do campo no plano cartesiano
left_x = -F.col('stadiumLength') / 2
right_x = F.col('stadiumLength') / 2

# Extremos superior e inferior do campo no plano cartesiano
top_y = F.col('stadiumWidth') / 2
bottom_y = -F.col('stadiumWidth') / 2

# Eixos x e y da posição da bola a partir dos dados de tracking
ball_x = F.get("ballsNorm", 0)["x"]
ball_y = F.get("ballsNorm", 0)["y"]

## 6.1. Métricas por zona do campo

- Métrica 1: Distância percorrida no campo (medida pela menor distância entre os escanteios do time com a posse até a bola) 
    - Hipótese: Quanto mais o jogador com posse percorrer o campo com a bola na direção do gol, maior a ameaça de gol por estar mais próximo dele.
    - Relação: Diretamente proporcional
- Métrica 2: Quantidade total de jogadores dos dois times (entre a bola e o gol)
    - Hipótese: Quanto MAIS jogadores dos dois times entre a bola e gol, MENOR a ameaça de gol por haver maior possibilidade de alguma ação defensiva e também por haver chances de um possível chute ser bloqueado.
    - Relação: Inversamente proporcional
- Métrica 3: Vantagem numérica do ataque em relação à defesa (entre a bola e o gol)
    - Hipótese: Quanto MAIOR a vantagem numérica do ataque em relação à defesa, MAIOR a ameaça de gol por ter maiores chance de ações ofensivas e menores chances de ações defensivas
    - Relação: Diretamente proporcional

In [21]:
ZONES = ['full', 'half', 'third_2', 'third_3']

df_games_events_players_ball_goal = add_zone_metrics(
    df_games_events_tracking_norm, ball_x, ball_y, left_x, right_x, top_y, bottom_y, ZONES
)

## 6.2. Normalização Min-Max e cálculo do threat score

In [22]:
# ============================================================
# Normalização Min-Max para cada zona (full, half, third_2, third_3)
# ============================================================

# Lista de (nome_da_coluna, inverter) para cada zona:
# - progression: diretamente proporcional (mais distância percorrida = mais ameaça)
# - total: inversamente proporcional (mais jogadores entre bola e gol = menos ameaça)
# - advantage: diretamente proporcional (mais vantagem numérica do ataque = mais ameaça)
norm_targets = []
for zone in ZONES:
    cols = zone_col_names(zone)
    norm_targets.append((cols['progression'], False))  # diretamente proporcional
    norm_targets.append((cols['total'], True))          # inversamente proporcional
    norm_targets.append((cols['advantage'], False))     # diretamente proporcional

# Monta os agregados de min/max de cada coluna acima (uma única passada no df)
agg_exprs = []
for col_name, _ in norm_targets:
    agg_exprs.append(F.min(col_name).alias(f'min_{col_name}'))
    agg_exprs.append(F.max(col_name).alias(f'max_{col_name}'))

# Executa a agregação (F.min/F.max ignoram os NULLs do mascaramento por zona)
stats = df_games_events_players_ball_goal.agg(*agg_exprs).first()

# Aplica a fórmula min-max (x - min) / (max - min) em cada coluna,
# invertendo (1 - minmax) quando a métrica for inversamente proporcional
norm_cols = {}
for col_name, invert in norm_targets:
    min_val = stats[f'min_{col_name}']
    max_val = stats[f'max_{col_name}']
    expr = (F.col(col_name) - F.lit(min_val)) / F.lit(max_val - min_val)
    if invert:
        expr = 1 - expr
    norm_cols[f'{col_name}_norm'] = F.round(expr, 3)

#df_threat = df_games_events_players_ball_goal.withColumns(norm_cols)

df_threat = (
    df_games_events_players_ball_goal
    .withColumns(norm_cols)
    # drop das colunas que foram normalizadas
    .drop(*[col_name for col_name, _ in norm_targets])
)

# Threat score por zona = média das 3 componentes normalizadas daquela zona
threat_score_cols = {}
for zone in ZONES:
    cols = zone_col_names(zone)
    suffix = '' if zone == 'full' else f'_{zone}'
    threat_score_cols[f'threat_score{suffix}'] = F.round(
        (
            F.col(f"{cols['progression']}_norm") +
            F.col(f"{cols['total']}_norm") +
            F.col(f"{cols['advantage']}_norm")
        ) / F.lit(3.0), 3
    )

df_threat_final = df_threat.withColumns(threat_score_cols)

# Checkpoint para truncar a lineage antes de seguir com as próximas análises
df_threat_final = df_threat_final.localCheckpoint()

In [23]:
df_threat_final.show(5)

+------+-------------+---------+--------------------+------+-----------------+------------+--------------------+--------------+-----------------------+--------+--------------------+-------------+----------------+-----------+--------------+-----------------------+-----------------------+---------------+----------+-------------+--------------+----------------+-----------------+---------------------+-------------+-------------+------------+----------------+-------------+----------------+-----------------------+------------------------+--------------------+--------------------+--------------------+---------+---------+--------------+--------------+-----------------+-----------------+-----------------+-----------------+---------------------+------------------+----------------------+--------------------------+-----------------------+---------------------------+-----------------------------+--------------------------+------------------------------+-----------------------------+----------------

# 7. Impacto da ameaça na posse (threat_score_impact)

Mede o efeito de cada evento dentro da sua posse: `threat_score` do evento seguinte na mesma posse menos o do próprio evento. Posses que terminam sem sequência (último evento) recebem impacto 0.

In [24]:
# threat_score_impact: efeito do evento atual, medido pelo threat_score do evento
# seguinte na mesma posse menos o threat_score do próprio evento.
# (partition por gameId + possession_id: possession_id reinicia a cada jogo, não é único globalmente)
w_possession = Window.partitionBy('gameId', 'possession_id').orderBy('startGameClock')

df_threat_impact = (
    df_threat_final
    .withColumn('threat_score_depois', F.lead('threat_score', 1).over(w_possession))
    .fillna(0, subset=['threat_score_depois'])
    .withColumn(
        'threat_score_impact',
        F.round(F.col('threat_score_depois') - F.col('threat_score'), 3)
    )
)

df_threat_impact.show(5)

+------+-------------+---------+--------------------+------+-----------------+---------+--------------------+--------------+-----------------------+--------+--------------------+-------------+-------------------+-----------+-------------+-----------------------+-----------------------+---------------+----------+-------------+--------------+----------------+-----------------+---------------------+-------------+-------------+------------+----------------+-------------+----------------+-----------------------+------------------------+--------------------+--------------------+--------------------+---------+---------+--------------+--------------+-----------------+-----------------+-----------------+-----------------+---------------------+------------------+----------------------+--------------------------+-----------------------+---------------------------+-----------------------------+--------------------------+------------------------------+-----------------------------+-----------------

# 8. Validações e checagens

## 8.1. Threat score nulo — quanto é explicado por bola fora do campo

In [25]:
df_null_threat = df_threat_impact.filter(F.col('threat_score').isNull())

ball_fora_do_campo = (
    (ball_x < left_x) |
    (ball_x > right_x) |
    (ball_y > top_y) |
    (ball_y < bottom_y)
)

df_null_threat_check = df_null_threat.select(
    'gameId',
    'eventId',
    'eventTypeDescription',
    ball_x.alias('ball_x'),
    ball_y.alias('ball_y'),
    left_x.alias('left_x'),
    right_x.alias('right_x'),
    top_y.alias('top_y'),
    bottom_y.alias('bottom_y'),
    (ball_x < left_x).alias('ball_x_menor_left_x'),
    (ball_x > right_x).alias('ball_x_maior_right_x'),
    (ball_y > top_y).alias('ball_y_maior_top_y'),
    (ball_y < bottom_y).alias('ball_y_menor_bottom_y'),
    ball_fora_do_campo.alias('ball_fora_do_campo'),
)

#df_null_threat_check.show(50, truncate=False)

total_nulls = df_null_threat.count()
fora_do_campo = df_null_threat_check.filter(F.col('ball_fora_do_campo')).count()

print(f'Total de threat_score nulos: {total_nulls}')
print(f'Explicados por bola fora do campo: {fora_do_campo} ({fora_do_campo / total_nulls:.2%})')
print(f'Não explicados: {total_nulls - fora_do_campo}')

Total de threat_score nulos: 371
Explicados por bola fora do campo: 371 (100.00%)
Não explicados: 0


## 8.2. Inspeção manual de uma partida e plots de posse (exploratório)

Células de sanity check pontual — não fazem parte do pipeline principal.

In [26]:
df_teste = (
    df_threat_impact
    .filter(F.col('gameId') == 4438)
    .select(
        'gameId', 
        'eventId',
        'period', 
        'startGameClock', 
        'flipped_homeTeam',
        'eventTypeDescription', 
        'eventSubTypeDescription', 
        'eventOutcomeDescription',  
        'homeTeam',
        'possession_id',         
        'threat_score',
        'threat_score_depois', 
        'threat_score_impact'
        )   
)

df_teste.show(2454, truncate=False)

#df_teste.toPandas().to_csv('eventos_partida_teste.csv')

+------+--------------------------------+------+--------------+----------------+--------------------+-----------------------+------------------------------+--------+-------------+------------+-------------------+-------------------+
|gameId|eventId                         |period|startGameClock|flipped_homeTeam|eventTypeDescription|eventSubTypeDescription|eventOutcomeDescription       |homeTeam|possession_id|threat_score|threat_score_depois|threat_score_impact|
+------+--------------------------------+------+--------------+----------------+--------------------+-----------------------+------------------------------+--------+-------------+------------+-------------------+-------------------+
|4438  |9e6a498f54bad910e13edda2bc83167a|1     |0             |false           |Pass                |Standard Pass          |Complete                      |false   |0            |0.339       |0.339              |0.0                |
|4438  |60717c0b4d7bb1d5770a3375098b553b|1     |0             |false

In [27]:
# # exemplo de clearance com posse mantida (sem posse invertida pra pegar ameaça do adversário) seguida de disputa 
# # + 
# # clearance seguida de posse adversária (com posse invertida pra pegar ameaça do adversário)
# df_teste.filter(F.col('possession_id').isin([177,178,179])).show(truncate=False)

# events = [
# 'fbac99eb7506525c391fafcc37842620',
# '2e2b0f61c95018afd1885afb52897086',
# 'd73ae92a5b5aa72b7749c79ac4aedebc',
# '8138df4d2e1eb56ad6cb48f58b91daba',
# '531990bbb811d311d476e9dd2e0ada20',
# '35b41b99cc2e1e79fa94170b63b6a564'
# ]

# for event in events:

#     df_event = df_threat_impact.filter(F.col("eventId") == event)

#     plot_threat_event(df_event, show_player_names=False)

In [28]:
# # exemplo de bola roubada seguida de Clearance que virou contra-ataque
# df_teste.filter(F.col('possession_id').isin([155,156])).show(truncate=False)

# events = [
# '14061ee291a01c235879bd85af334ea0',
# 'b0773bd17ac0d55ee397055fc5000549',
# '14061ee291a01c235879bd85af334ea0'
# ]

# for event in events:

#     # df_event = df_threat_impact.filter(F.col("eventId") == event)

#     # plot_threat_event(df_event, show_player_names=False)

# 9. Persistência do dataset final

Grava `df_threat_impact` (eventos + threat_score + threat_score_impact) em parquet, pronto pra ser consumido nas próximas etapas de feature engineering.

In [29]:
output_path = str(Path().resolve().parent.parent / "data" / "threat_dataset")
df_threat_impact.write.mode("overwrite").parquet(output_path)